# NLP - DistilBERT multilingual + XGboost

- Nivel 1: Few-Shot - Aqui, além do texto, fornecemos ao modelo alguns exemplos de tickets já classificados, dentro da propria base. 

In [ ]:
# Imports
# ------------------------------
#import sys
#!{sys.executable} -m pip install tqdm
#!pip install numpy==1.25.2
#!pip uninstall torch -y
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize

# ----------------------------



In [ ]:
# --- 1️⃣ Ler o arquivo Parquet ---
arquivo_parquet = "ifpr_helpdesk_completo.parquet"
df_filtrado = pd.read_parquet(arquivo_parquet)

# --- 2️⃣ Visualizar as primeiras linhas ---
df_filtrado.head()


,Ticker,Requerente,Grupo,Categoria,Data Abertura,Data Solução,Data Fechamento,Atendente,Satisfação,Data Iteração,Usuário Iteração,Resolvido,texto
0,16749,ELIDIONETE DE ANDRADE,SIG_TECNICO,-,29/12/2017 16:43,09/01/2018 11:30,11/01/2018 10:21,TATIANA MAYUMI NIWA,-,04/01/2018 08:42,ELIDIONETE DE ANDRADE,sim,16749 ELIDIONETE DE ANDRADE
1,16749,ELIDIONETE DE ANDRADE,SIGAA,-,29/12/2017 16:43,09/01/2018 11:30,11/01/2018 10:21,TATIANA MAYUMI NIWA,-,04/01/2018 08:42,ELIDIONETE DE ANDRADE,sim,16749 ELIDIONETE DE ANDRADE
2,16748,LARISSA DINIZ RIBEIRO,SIGAA,-,29/12/2017 14:22,02/01/2018 09:32,04/01/2018 08:29,TATIANA MAYUMI NIWA,-,-,-,sim,16748 LARISSA DINIZ RIBEIRO
3,16747,FABIANA APARECIDA PEREIRA DA SILVA,SIG_TECNICO,-,29/12/2017 11:55,02/02/2018 13:57,02/02/2018 13:57,LUCIANA WISTUBA COSMO DE SIQUEIRA E SILVA,-,29/12/2017 13:20,FABIANA APARECIDA PEREIRA DA SILVA,sim,16747 FABIANA APARECIDA PEREIRA DA SILVA
4,16747,FABIANA APARECIDA PEREIRA DA SILVA,SIGAA,-,29/12/2017 11:55,02/02/2018 13:57,02/02/2018 13:57,LUCIANA WISTUBA COSMO DE SIQUEIRA E SILVA,-,29/12/2017 13:20,FABIANA APARECIDA PEREIRA DA SILVA,sim,16747 FABIANA APARECIDA PEREIRA DA SILVA


In [12]:
# --- 3️⃣ Informações gerais do DataFrame ---
print(df_filtrado.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41648 entries, 0 to 41647
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Ticker            41648 non-null  int64 
 1   Requerente        41648 non-null  object
 2   Grupo             41648 non-null  object
 3   Categoria         41648 non-null  object
 4   Data Abertura     41648 non-null  object
 5   Data Solução      41648 non-null  object
 6   Data Fechamento   41648 non-null  object
 7   Atendente         41648 non-null  object
 8   Satisfação        41648 non-null  object
 9   Data Iteração     41648 non-null  object
 10  Usuário Iteração  41648 non-null  object
 11  Resolvido         41648 non-null  object
 12  texto             41648 non-null  object
dtypes: int64(1), object(12)
memory usage: 4.1+ MB
None


In [13]:
# --- 4️⃣ Estatísticas básicas (opcional) ---
df_filtrado.describe(include='all')

,Ticker,Requerente,Grupo,Categoria,Data Abertura,Data Solução,Data Fechamento,Atendente,Satisfação,Data Iteração,Usuário Iteração,Resolvido,texto
count,41648.000000,41648,41648,41648,41648,41648,41648,41648,41648,41648,41648,41648,41648
unique,NaN,2233,76,32,38202,34054,15655,808,7,19060,733,2,39507
top,NaN,TATIANE VARELA,Suporte Reitoria,-,01/06/2022 09:00,-,-,GUSTAVO COSTA BRAUNER,-,-,-,sim,37211 ONIVALDO FLORES JUNIOR
freq,NaN,686,7339,34936,9,2367,2418,3460,40440,20201,20203,39230,5
mean,29045.946264,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,11423.931902,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,9460.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,19022.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,28950.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,38949.250000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_total = df_filtrado
#para limitar linhas e testar 
#df_total = df_total.head(10000)
print(f"Tamanho da base limitada: {len(df_total)} linhas")

Tamanho da base limitada: 41648 linhas


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
# ------------------------------
# Prepara labels
# ------------------------------
if 'Grupo' in df_total.columns:
    le = LabelEncoder()
    df_total['Grupo_encoded'] = le.fit_transform(df_total['Grupo'])
else:
    df_total['Grupo_encoded'] = -1  # placeholder se não tiver labels

# ------------------------------
# Inicializa modelo de embeddings
# ------------------------------
model = SentenceTransformer('distiluse-base-multilingual-cased-v2')

# ------------------------------
# Gera embeddings
# ------------------------------
texts = df_total['texto'].astype(str).tolist()
print("Gerando embeddings...")
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)
embeddings = np.array(embeddings)

# ------------------------------
#  treino/teste (sem stratify)
# ------------------------------
if df_total['Grupo_encoded'].nunique() > 1:
    X_train, X_test, y_train, y_test = train_test_split(
        embeddings,
        df_total['Grupo_encoded'],
        test_size=0.2,
        random_state=42,
        stratify=None  
    )

    # ------------------------------
    # Treinar classificador (Logistic Regression)
    # ------------------------------
    clf = LogisticRegression(max_iter=1000)
    print("Treinando classificador...")
    clf.fit(X_train, y_train)

    # ------------------------------
    # Predição completa (todas as amostras)
    # ------------------------------
    print("Predizendo grupos...")
    preds_rgl = []
    for emb in tqdm(embeddings, desc="Predizendo"):
        pred = clf.predict([emb])[0]
        preds_rgl.append(pred)

    df_total['pred_distilbert_RGLogistic'] = le.inverse_transform(preds_rgl)

    # ------------------------------
    # Métricas de avaliação
    # ------------------------------
    y_true = df_total['Grupo_encoded']

    # Accuracy
    accuracy_rgl = accuracy_score(y_true, preds_rgl)
    print("Accuracy (rgl):", accuracy_rgl)

    # F1 Score (macro e weighted)
    f1_macro_rgl = f1_score(y_true, preds_rgl, average='macro')
    f1_weighted_rgl = f1_score(y_true, preds_rgl, average='weighted')
    print("F1 Macro (rgl):", f1_macro_rgl)
    print("F1 Weighted (rgl):", f1_weighted_rgl)

    # AUC Multiclasse (One-vs-Rest)
    y_bin = label_binarize(y_true, classes=np.unique(y_true))
    preds_bin = label_binarize(preds_rgl, classes=np.unique(y_true))
    if y_bin.shape[1] > 1:  # precisa ter mais de uma classe
        auc_rgl = roc_auc_score(y_bin, preds_bin, average='macro')
        print("AUC Macro (rgl):", auc_rgl)
    else:
        print("AUC não calculada: menos de 2 classes presentes.")

    # Classification report
    #print("Classification Report (rgl):")
    #print(classification_report(y_true, preds_rgl, zero_division=0))


else:
    print("Não há classes suficientes para treino.")
    df_total['pred_distilbert_RGLogistic'] = None

# ------------------------------
# Salva resultados
# ------------------------------
saida_parquet = "ifpr_helpdesk_classificado_distilbert_rgl.parquet"
df_total.to_parquet(saida_parquet, index=False)
print(f"Nível 1 concluído! Arquivo salvo: {saida_parquet}")

# ------------------------------
# Visualiza resultados brevemente
# ------------------------------

print(df_total['pred_distilbert_RGLogistic'].value_counts())


Gerando embeddings...


Batches:   0%|          | 0/1302 [00:00<?, ?it/s]

C:\Users\jgeov\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\models\distilbert\modeling_distilbert.py:392: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Treinando classificador...
Predizendo grupos...


Predizendo: 100%|██████████| 41648/41648 [00:15<00:00, 2702.35it/s]


Accuracy (rgl): 0.33463791778716867
F1 Macro (rgl): 0.07102331168747304
F1 Weighted (rgl): 0.29476253802548213
AUC Macro (rgl): 0.5284756684868879
Nível 1 concluído! Arquivo salvo: ifpr_helpdesk_classificado_distilbert_rgl.parquet
pred_distilbert_RGLogistic
Suporte Reitoria           16921
SIPAC                      10356
RTIC-PALMAS                 3422
SIGAE - Negocial            2778
SEI                         2500
SIGAA                       1240
Infraestrutura              1085
RTIC-CELVVD                  691
Manutenção predial           654
-                            593
RTIC-CTBA                    577
Suporte Informática          540
RTIC-CPLG                    113
RTIC-Dom Bosco                56
SIG_NEGOCIAL                  45
SEI CPAD                      40
Infraestrutura-Umuarama       23
RTIC - Assis                  12
Banco de Dados                 2
Name: count, dtype: int64


In [ ]:

# ------------------------------
# Tratar classes raras
# ------------------------------
min_samples = 2
counts = df_total['Grupo'].value_counts()
valid_classes = counts[counts >= min_samples].index

# Classes raras viram "Outros"
df_total['Grupo_tratado'] = df_total['Grupo'].apply(lambda x: x if x in valid_classes else "Outros")

# LabelEncoder final
le = LabelEncoder()
df_total['Grupo_encoded'] = le.fit_transform(df_total['Grupo_tratado'])

# ------------------------------
# Inicializa modelo de embeddings (DistilBERT)
# ------------------------------
model = SentenceTransformer('distiluse-base-multilingual-cased-v2')

# ------------------------------
# Gera embeddings
# ------------------------------
texts = df_total['texto'].astype(str).tolist()
print("Gerando embeddings...")
embeddings = np.array(model.encode(texts, show_progress_bar=True, batch_size=32))

# ------------------------------
#  treino/teste (sem stratify - evita erro)
# ------------------------------
if df_total['Grupo_encoded'].nunique() > 1:
    X_train, X_test, y_train, y_test = train_test_split(
        embeddings,
        df_total['Grupo_encoded'],
        test_size=0.2,
        random_state=42,
        stratify=None
    )

    # ------------------------------
    # Treinar classificador XGBoost
    # ------------------------------
    clf = XGBClassifier(
        objective='multi:softmax',
        eval_metric='mlogloss',
        use_label_encoder=False,
        random_state=42
    )
    print("Treinando classificador XGBoost...")
    clf.fit(X_train, y_train)

    # ------------------------------
    # Predição completa (todas as amostras)
    # ------------------------------
    print("Predizendo grupos...")
    preds_xgb_enc = clf.predict(embeddings)
    df_total['pred_distilbert_XGB'] = le.inverse_transform(preds_xgb_enc)

    # ------------------------------
    # Métricas de avaliação
    # ------------------------------
    y_true = df_total['Grupo_encoded']

    # Accuracy
    accuracy_xgb = accuracy_score(y_true, preds_xgb_enc)
    print("Accuracy (XGB):", accuracy_xgb)

    # F1 Score (macro e weighted)
    f1_macro_xgb = f1_score(y_true, preds_xgb_enc, average='macro')
    f1_weighted_xgb = f1_score(y_true, preds_xgb_enc, average='weighted')
    print("F1 Macro (XGB):", f1_macro_xgb)
    print("F1 Weighted (XGB):", f1_weighted_xgb)

    # AUC Multiclasse (One-vs-Rest)
    y_bin = label_binarize(y_true, classes=np.unique(y_true))
    preds_bin = label_binarize(preds_xgb_enc, classes=np.unique(y_true))
    if y_bin.shape[1] > 1:
        auc_xgb = roc_auc_score(y_bin, preds_bin, average='macro')
        print("AUC Macro (XGB):", auc_xgb)
    else:
        print("AUC não calculada: menos de 2 classes presentes.")

    # Classification report
    #print("Classification Report (XGB):")
    #print(classification_report(y_true, preds_xgb_enc, zero_division=0))


else:
    print("Não há classes suficientes para treino.")
    df_total['pred_distilbert_XGB'] = None

# ------------------------------
# Salva resultados
# ------------------------------
saida_parquet = "ifpr_helpdesk_classificado_distilbert_xgb.parquet"
df_total.to_parquet(saida_parquet, index=False)
print(f"Nível 1 concluído! Arquivo salvo: {saida_parquet}")

# ------------------------------
# Visualiza resultados brevemente
# ------------------------------
df_total['pred_distilbert_XGB'].value_counts()


Gerando embeddings...


Batches:   0%|          | 0/1302 [00:00<?, ?it/s]

Treinando classificador XGBoost...


C:\Users\jgeov\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [00:05:05] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Predizendo grupos...
Accuracy (XGB): 0.8480119093353823
F1 Macro (XGB): 0.8175684385124999
F1 Weighted (XGB): 0.84964021948742
AUC Macro (XGB): 0.8997815824325576
Nível 1 concluído! Arquivo salvo: ifpr_helpdesk_classificado_distilbert_xgb.parquet


pred_distilbert_XGB
Suporte Reitoria                                                  8366
SIPAC                                                             6215
RTIC-PALMAS                                                       3244
SIGAA                                                             3103
SEI                                                               2802
SIGAE - Negocial                                                  2266
Infraestrutura                                                    2000
RTIC-CTBA                                                         1850
-                                                                 1412
Manutenção predial                                                1289
RTIC-CPLG                                                          829
Sistemas                                                           809
Problemas GLPI                                                     718
RTIC-Dom Bosco                                           